In [ ]:
import os

# Work around Colab/NVML allocator failures before importing torch.
os.environ["PYTORCH_NO_CUDA_NVML"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "backend:cudaMallocAsync"


# CXR Second Reader — Qwen3-VL report generation

This local-first notebook continues the Indiana University chest X-ray project with a public Qwen3-VL model. It includes data preparation, study-level splits, a zero-shot validation baseline, resumable QLoRA training, and a paired final test evaluation.

## Run in stages

1. Connect VS Code to a local Python 3.11 Jupyter kernel with CUDA-enabled PyTorch.
2. Set `CXR_DATASET_ROOT` to the already extracted Indiana dataset and `CXR_SHARED_DIR` to a project data folder.
3. Run data preparation through **Save splits**. These split CSVs and the manifest are reused on later runs.
4. Install the remaining model dependencies, confirm the GPU, load the public base model, and inspect a validation preview.
5. Run the training-data sanity check, train in resumable optimizer-update chunks, and compare checkpoints on validation only.
6. After model selection is complete, run the paired held-out evaluation once.

Do not use **Run all**: it can start training and evaluation. The model downloads to the local Hugging Face cache; the dataset is not downloaded unless `CXR_ALLOW_DATASET_DOWNLOAD=1` is explicitly set. Checkpoints and metrics save under `QWEN_SAVE_DIR` (defaults to `<project>/qwen3vl_cxr`).

This is an educational research prototype, not for clinical use. Text-overlap scores do not establish clinical correctness.


# Indiana University Chest X-rays — Data Preparation

This notebook prepares the Indiana University Chest X-ray dataset for a medical Vision-Language Model (VLM).

Workflow:
1. Brief dataset overview & EDA
2. Data Cleaning & Image-Report Mapping
3. Image Preprocessing & Quality Checks
4. Online Data Augmentation


## Import needed libraries

In [ ]:
!pip -q install missingno seaborn kagglehub opencv-python


In [ ]:
import pandas as pd
import numpy as np
import random
from collections import Counter
import re
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import missingno as msno
import textwrap
from pathlib import Path
from IPython.display import display
from PIL import Image
import PIL.Image
import cv2
import os
import kagglehub


## Dataset Overview


In [ ]:
# Use an existing extracted dataset by default; downloading requires explicit opt-in.
import os
from pathlib import Path
if os.environ.get("CXR_DATASET_ROOT"):
    dataset_path = Path(os.environ["CXR_DATASET_ROOT"]).expanduser()
    if not dataset_path.is_dir():
        raise FileNotFoundError(f"CXR_DATASET_ROOT does not exist: {dataset_path}")
elif os.environ.get("CXR_ALLOW_DATASET_DOWNLOAD") == "1":
    dataset_path = Path(kagglehub.dataset_download("raddar/chest-xrays-indiana-university"))
else:
    raise RuntimeError(
        "Set CXR_DATASET_ROOT to the existing Indiana dataset folder. "
        "To allow KaggleHub to download it, explicitly set CXR_ALLOW_DATASET_DOWNLOAD=1."
    )
print("Indiana dataset:", dataset_path)


In [ ]:
# KaggleHub returns the actual Colab cache directory; do not search Kaggle-only paths.
INPUT_DIR = Path(dataset_path)

report_files = list(INPUT_DIR.rglob("indiana_reports.csv"))
projection_files = list(INPUT_DIR.rglob("indiana_projections.csv"))
if not report_files or not projection_files:
    raise FileNotFoundError(f"Indiana metadata CSVs were not found under {INPUT_DIR}")

REPORTS_PATH = report_files[0]
PROJECTIONS_PATH = projection_files[0]
image_dirs = [p for p in INPUT_DIR.rglob("images_normalized") if p.is_dir()]
if not image_dirs:
    image_dirs = [p for p in INPUT_DIR.rglob("images") if p.is_dir()]
if not image_dirs:
    raise FileNotFoundError(f"No images or images_normalized directory under {INPUT_DIR}")
IMAGES_DIR = image_dirs[0]

reports = pd.read_csv(REPORTS_PATH)
projections = pd.read_csv(PROJECTIONS_PATH)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
image_paths = sorted(
    p for p in IMAGES_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)
print(f"Medical reports: {len(reports):,}")
print(f"Projection records: {len(projections):,}")
print(f"Image files found: {len(image_paths):,}")


## EDA & Cleaning

In [ ]:
reports.head()

In [ ]:
reports.info()

In [ ]:
reports.isnull().sum()

In [ ]:
reports.duplicated().sum()

In [ ]:
projections.head()

In [ ]:

projections.info()

In [ ]:
projections.duplicated().sum()

### Merging Projections and Reports

In [ ]:
df = pd.merge(reports, projections, on='uid', how='inner')
df.head()

In [ ]:
df.info()

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage': (df.isnull().sum() / len(df)) * 100
})

print(missing)

In [ ]:
filenames_no_findings = df.loc[
    df["findings"].fillna("").str.strip().eq(""), "filename"
]

print(f"Number of files with no findings: {len(filenames_no_findings)}")
display(filenames_no_findings)

In [ ]:
# Heatmap showing correlation of missingness
msno.heatmap(df)
plt.show()

In [ ]:
# Keep missing report fields empty; placeholders would become false training targets.
df = df.copy()
for col in ["findings", "impression", "indication"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()


In [ ]:
df.isnull().sum()

### Actual image files with file names

In [ ]:
# Identify actual image files using the paths collected in the overview
actual_filenames = {path.name for path in image_paths}
print(f"Number of image files found: {len(actual_filenames)}")

### Matching Projection Metadata

In [ ]:
# Keep only images that have matching projection metadata
valid_projections = df[
    df["filename"].isin(actual_filenames)
]
print(f"Records with matching projection metadata: {len(valid_projections)}")

In [ ]:
# Exclude images without metadata
excluded_images = sorted(
    actual_filenames - set(df["filename"].dropna())
)
print(f"Images excluded (no matching metadata): {len(excluded_images)}")

In [ ]:
# Create a DataFrame of excluded records
excluded_records = pd.DataFrame({
    "filename": excluded_images,
    "reason": "No matching projection metadata"
})

excluded_records

In [ ]:
valid_projections.info()

### Samples

In [ ]:

# Create a lookup dictionary: filename -> full path
image_lookup = {path.name: path for path in image_paths}

for i in range(min(6, len(valid_projections))):
    sample = valid_projections.iloc[i]
    filename = sample["filename"]
    # Find corresponding image
    img_path = image_lookup.get(filename)
    img = PIL.Image.open(img_path)
    width, height = img.size
    # Build the raw text content
    raw_text = f"Sample {i}\n" + "="*30 + "\n\n"
    for col, value in sample.items():
        raw_text += f"{col} : {value}\n\n"
    raw_text += f"Image path: {img_path}\n"
    raw_text += f"Image dimensions: {width} x {height}"
    # Wrap long lines
    wrapped_lines = []
    for line in raw_text.split('\n'):
        if len(line) > 60:
            wrapped_lines.extend(textwrap.wrap(line, width=60))
        else:
            wrapped_lines.append(line)
    text_content = '\n'.join(wrapped_lines)
    # Create a wider figure with custom width ratios between text and image
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1.3, 1]})
    # Left subplot: Display wrapped text metadata
    axes[0].axis("off")
    axes[0].text(
        0.0, 1.0,
        text_content,
        fontsize=9,
        verticalalignment='top',
        family='monospace'
    )
    # Right subplot: Display image
    axes[1].imshow(img, cmap='gray')
    axes[1].axis("off")
    axes[1].set_title(filename)
    # Adjust spacing
    fig.subplots_adjust(left=0.05, right=0.95, top=0.92, bottom=0.05, wspace=0.1)
    plt.show()

In [ ]:

valid_projections['projection'].value_counts(normalize=True)

In [ ]:
# See which projections exist for each UID
uid_views = (
    valid_projections
    .groupby("uid")["projection"]
    .agg(lambda x: set(x))
)

# UIDs with only Frontal images
frontal_only_uids = uid_views[uid_views == {"Frontal"}].index

# UIDs with only Lateral images
lateral_only_uids = uid_views[uid_views == {"Lateral"}].index

print(f"Frontal-only UIDs: {len(frontal_only_uids)}")
print(f"Lateral-only UIDs: {len(lateral_only_uids)}")

In [ ]:
valid_projections['Problems'].unique()

### Text Length frequencies

In [ ]:
exclude_cols = ["uid", "projection", "filename"]
cols_to_plot = [col for col in valid_projections.columns if col not in exclude_cols][:7]

# Create a subplot grid
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
  lengths = valid_projections[col].astype(str).str.len()

  # Plot on the specific subplot axis
  sns.histplot(lengths, bins=30, kde=True, ax=axes[i])
  axes[i].set_title(f"Distribution of Text Length in '{col}'", fontsize=12)
  axes[i].set_xlabel("Text Length", fontsize=10)
  axes[i].set_ylabel("Count", fontsize=10)
  axes[i].grid(axis="y", linestyle="--", alpha=0.7)

# Hide any unused axes
for j in range(len(cols_to_plot), len(axes)):
  fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
problem_counts = valid_projections['Problems'].value_counts().head(15)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x=problem_counts.values,
    y=problem_counts.index,
    hue=problem_counts.index,
    palette='coolwarm',
    legend=False
)
plt.xlabel('Count of Problem', fontsize=12)
plt.ylabel('Problem report', fontsize=12)
plt.title('Top 15 Problems Distribution', fontsize=14, fontweight='bold')
plt.yticks(rotation=0)
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(
            str(int(width)),
            (width, p.get_y() + p.get_height() / 2.),
            xytext=(5, 0),
            textcoords="offset points",
            ha='left',
            va='center'
        )

plt.tight_layout()
plt.show()

## Image Preprocessing & Quality Checks

In this step, we inspect the X-ray images to ensure they are valid and suitable for further processing.

* Check image readability and file integrity.
* Identify corrupted or unsupported images.
* Analyze image dimensions and color modes.

**Note:** Original images will remain unchanged. Resizing and normalization will be performed later according to the selected model architecture.


In [ ]:
image_quality_results = []

for filename in valid_projections["filename"].drop_duplicates():
    image_path = image_lookup.get(filename)

    if image_path is None:
        image_quality_results.append({
            "filename": filename,
            "status": "Missing",
            "width": None,
            "height": None,
            "mode": None
        })
        continue

    try:
        with Image.open(image_path) as img:
            img.verify()

        with Image.open(image_path) as img:
            width, height = img.size
            mode = img.mode

        image_quality_results.append({
            "filename": filename,
            "status": "Valid",
            "width": width,
            "height": height,
            "mode": mode
        })

    except Exception:
        image_quality_results.append({
            "filename": filename,
            "status": "Corrupted",
            "width": None,
            "height": None,
            "mode": None
        })

image_quality_df = pd.DataFrame(image_quality_results)

print("Total unique mapped images:", len(image_quality_df))
print("\nImage status:")
print(image_quality_df["status"].value_counts())

print("\nImage modes:")
print(image_quality_df["mode"].value_counts(dropna=False))

print("\nImage dimensions summary:")
display(image_quality_df[["width", "height"]].describe())

print("\nSample results:")
display(image_quality_df.head())


### Image Augmentation



In [ ]:
def enhance_medical_image(img):
    # 1. Apply CLAHE for local contrast
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    st1 = clahe.apply(img)

    # 2. Apply gentle sharpening
    gaussian = cv2.GaussianBlur(st1, (0, 0), sigmaX=1.5)
    sharpened = cv2.addWeighted(st1, 1.3, gaussian, -0.3, 0)

    return sharpened

In [ ]:
random.seed(42)
image_path = random.choice(image_paths)

# Load image in grayscale (matching your dataset's 'L' mode)
original_img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# Process the image
enhanced_img = enhance_medical_image(original_img)

# Plot Before and After side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Before Plot
axes[0].imshow(original_img, cmap='gray')
axes[0].set_title('Before (Original Grayscale)', fontsize=14, fontweight='bold')
axes[0].axis('off')

# After Plot
axes[1].imshow(enhanced_img, cmap='gray')
axes[1].set_title(
    'After (CLAHE + Edge Sharpening)', fontsize=14, fontweight='bold'
)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Data Filtering

In [ ]:
valid_projections = valid_projections.copy()
for col in ["findings", "impression", "indication"]:
    if col in valid_projections.columns:
        valid_projections[col] = valid_projections[col].fillna("").astype(str).str.strip()

has_report = (
    valid_projections["findings"].ne("")
    | valid_projections["impression"].ne("")
)
df_new = valid_projections.loc[has_report].copy()


In [ ]:
def make_target(row):
    sections = []
    if row["findings"]:
        sections.append("Findings: " + row["findings"])
    if row["impression"]:
        sections.append("Impression: " + row["impression"])
    return " ".join(sections)

df_new["report_combined"] = df_new.apply(make_target, axis=1)
df_new["report_length"] = df_new["report_combined"].str.len()
df_new["word_count"] = df_new["report_combined"].str.split().str.len()
print("Usable reports:", len(df_new))
display(df_new[["uid", "filename", "report_combined"]].head())


In [ ]:
sample_report_combined = df_new['report_combined'].sample(40, random_state=42)
print("\nSample 'report_combined' Entries:\n")
for idx, combine in enumerate(sample_report_combined):
    print(f"{idx + 1}: {combine}\n")

In [ ]:
# Plotting the distribution of report lengths
plt.figure(figsize=(14, 7))
sns.histplot(df_new['report_length'], bins=50, kde=True, color='dodgerblue')
plt.title("Distribution of Concatenated Report Lengths")
plt.xlabel("Report Length (Number of Characters)")
plt.ylabel("Frequency")
plt.axvline(df_new['report_length'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['report_length'].mean():.2f}")
plt.axvline(df_new['report_length'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['report_length'].median():.2f}")
plt.legend()
plt.tight_layout()
plt.show()

# Display summary statistics
print("\nSummary statistics for 'report_length':\n")
print(df_new['report_length'].describe())

In [ ]:
q90 = df_new['report_length'].quantile(0.90)
q95 = df_new['report_length'].quantile(0.95)
q99 = df_new['report_length'].quantile(0.99)

print("\nLength Threshold Suggestions:")
print(f"90th Percentile: {q90} characters")
print(f"95th Percentile: {q95} characters")
print(f"99th Percentile: {q99} characters")
print(f"Mean Length: {df_new['report_length'].mean():.2f} characters")
print(f"Median Length: {df_new['report_length'].median():.2f} characters")

In [ ]:
df_new['word_count'] = df_new['report_combined'].apply(lambda x: len(x.split()))

# Display a few samples to verify
print("\nSample word counts:\n")
print(df_new[['report_combined', 'word_count']].head(10))

In [ ]:
plt.figure(figsize=(14, 7))
sns.histplot(df_new['word_count'], bins=50, kde=True, color='coral')
plt.title("Distribution of Word Count in Combined Reports")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.axvline(df_new['word_count'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['word_count'].mean():.2f}")
plt.axvline(df_new['word_count'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['word_count'].median():.2f}")
plt.legend()
plt.tight_layout()
plt.show()

# Display summary statistics
print("\nSummary statistics for 'word_count':\n")
print(df_new['word_count'].describe())

In [ ]:
print(f"90th Percentile Length Threshold: {q90} characters")

# Initial number of rows before filtering
initial_rows = len(df_new)

# Apply filtering
df_new = df_new[df_new['report_length'] <= q90]

# New number of rows after filtering
filtered_rows = len(df_new)

print(f"Number of rows after filtering: {filtered_rows}")
print(f"Number of rows removed: {initial_rows - filtered_rows}")

In [ ]:
plt.figure(figsize=(14, 7))
sns.histplot(df_new['report_length'], bins=50, kde=True, color='dodgerblue')
plt.title("Distribution of Report Lengths After Removing 90th Percentile")
plt.xlabel("Report Length (Number of Characters)")
plt.ylabel("Frequency")
plt.axvline(df_new['report_length'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['report_length'].mean():.2f}")
plt.axvline(df_new['report_length'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['report_length'].median():.2f}")
plt.legend()
plt.tight_layout()

plt.figure(figsize=(14, 7))
sns.histplot(df_new['word_count'], bins=50, kde=True, color='coral')
plt.title("Distribution of Word Count After Removing 90th Percentile")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.axvline(df_new['word_count'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['word_count'].mean():.2f}")
plt.axvline(df_new['word_count'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['word_count'].median():.2f}")
plt.legend()
plt.tight_layout()
plt.show()

# Display summary statistics
print("\nSummary statistics for 'word_count':\n")
print(df_new['word_count'].describe())

In [ ]:
sample_report_combined = df_new['report_combined'].sample(20, random_state=42)
print("\nSample 'report_combined' Entries:\n")
for idx, combine in enumerate(sample_report_combined):
    print(f"{idx + 1}: {combine}\n")

## Preprocess "report_combines"

In [ ]:
def preprocess_report_v2(report):
    if pd.isna(report):
        return ""
    # Preserve numbers, punctuation, negation, and laterality for report generation.
    report = re.sub(r"\bX{3,}\b", " ", str(report), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", report).strip()

df_new["processed_report"] = df_new["report_combined"].map(preprocess_report_v2)
df_new = df_new[df_new["processed_report"].str.strip().ne("")].copy()
display(df_new[["report_combined", "processed_report"]].head())


In [ ]:
sample_processed_report = df_new['processed_report'].sample(20, random_state=69)
print("\nSample 'processed_report' Entries:\n")
for idx, combine in enumerate(sample_processed_report):
    print(f"{idx + 1}: {combine}\n")

In [ ]:
all_words = " ".join(df_new['processed_report']).split()

# Count unique words and tags
unique_words = set(all_words)
word_count = len(unique_words)
print(f"Total unique words: {word_count}")

# Frequency distribution of words
word_freq = Counter(all_words)

# Top 10 most common words
print("\nTop 30 most common words:")
print(word_freq.most_common(30))

### Re-exmaining mesh

In [ ]:
# Check a few examples to confirm delimiter patterns
print("\nSample MeSH values:\n", df_new['MeSH'].head(5))

# Split by comma, semicolon, or other delimiters
mesh_terms = df_new['MeSH'].apply(lambda x: re.split(r'[;,/]', x.strip().lower()))

# Flatten the list of lists and count frequencies
all_terms = [term.strip() for sublist in mesh_terms for term in sublist if term.strip()]
term_counts = Counter(all_terms)

# Display the 15 most common terms
common_terms = term_counts.most_common(15)
print("\nTop 15 Most Common MeSH Terms:\n", common_terms)

# Plot the term frequency
terms, counts = zip(*common_terms)
plt.figure(figsize=(14, 7))
sns.barplot(x=list(counts), y=list(terms), palette="coolwarm")
plt.title("Top 15 Most Frequent MeSH Terms")
plt.xlabel("Frequency")
plt.ylabel("MeSH Terms")
plt.tight_layout()
plt.show()

### Distribution Balance

In [ ]:
normal_keywords = {"normal", "unremarkable", "clear", "stable"}
non_normal_keywords = {
    "disease", "effusion", "nodule", "infiltrate", "pneumothorax", "opacity",
    "abnormality", "enlargement", "calcification", "atelectasis",
    "consolidation", "mass", "lesion",
}

def classify_mesh(mesh_terms):
    text = re.sub(r"[^a-z0-9]+", " ", str(mesh_terms).lower()).strip()
    tokens = set(text.split())
    if "within normal limits" in text or tokens.intersection(normal_keywords):
        return "normal"
    if tokens.intersection(non_normal_keywords):
        return "non-normal"
    return "non-normal"

df_new["class"] = df_new["MeSH"].apply(classify_mesh)
print("Class distribution:\n", df_new["class"].value_counts())


In [ ]:
# Plotting as a Pie Chart to show dristribution
class_counts = df_new['class'].value_counts()
plt.figure(figsize=(6, 6))
colors = sns.color_palette('Set2')[0:2]

plt.pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',  # Automatically calculates and displays percentages
    startangle=140,
    colors=colors,
    textprops={'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},  # Adds a clean border
)

plt.title(
    'Distribution: Normal vs. Other Problems', fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
df_new.info()

In [ ]:
df_new.head()

In [ ]:
df_new['comparison'].unique()

In [ ]:
df_new['comparison'].value_counts()

In [ ]:
df_new['Problems'].unique()

In [ ]:
df_new['MeSH'].unique()

In [ ]:
df_new['indication'].unique()

In [ ]:
df_new = df_new.drop(
    columns=["comparison", "image", "report_combined", "report_length", "word_count", "findings", "impression"],
    errors="ignore",
)


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return text
    text = str(text).lower()
    # Remove anonymization tokens like XXX, XXXX, XXXXX
    text = re.sub(r'\bx{3,}\b', '', text)
    # Remove punctuation
    text = re.sub(r'[^\w\s;/]', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
df_new['indication'] = df_new['indication'].apply(clean_text)

In [ ]:
df_new['Problems'] = df_new['Problems'].apply(clean_text)

In [ ]:
df_new['MeSH'] = df_new['MeSH'].apply(clean_text)

In [ ]:
df_new.head()

### Report

In [ ]:
def image_report(df_new, num_samples):
    """
    Visualize up to 1 or 2 images per report alongside the report text
    using the pre-declared image_lookup dictionary.
    """
    samples = df_new.sample(num_samples, random_state=42)

    for _, row in samples.iterrows():
        uid = row['uid']
        report = row['processed_report']

        # Get up to 2 image entries for this UID from the dataframe
        image_entries = df_new[df_new['uid'] == uid]['filename'].tolist()[:2]

        # Map directly using the pre-existing global image_lookup
        img_paths = [image_lookup[Path(img_entry).name] for img_entry in image_entries]

        num_images = len(img_paths)
        if num_images == 0:
            print(f"No images found for UID: {uid}")
            continue

        fig, axes = plt.subplots(1, num_images, figsize=(6 * num_images, 6))
        fig.suptitle(f"UID: {uid}\nReport: {report}", fontsize=10, ha='center', wrap=True)

        if num_images == 1:
            axes = [axes]

        for idx, img_path in enumerate(img_paths):
            original_img = cv2.imread(str(img_path),cv2.IMREAD_GRAYSCALE)
            image = enhance_medical_image(original_img)

            axes[idx].imshow(image, cmap='gray')
            axes[idx].axis('off')
            axes[idx].set_title(f"Image {idx + 1}")

        plt.tight_layout()
        plt.show()

In [ ]:
image_report(df_new, num_samples=2)

## Splitting

In [ ]:
# Persist the study-level split manifest locally for reproducible VS Code/Jupyter reruns.
from pathlib import Path
import os
import numpy as np
import pandas as pd
SPLIT_DIR = Path(os.environ.get("CXR_SHARED_DIR", REPO_ROOT / "cxr_shared")).expanduser()
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = SPLIT_DIR / "split_manifest.csv"
split_names = ["train", "validation", "test"]
split_ratios = {"train": 0.80, "validation": 0.10, "test": 0.10}
current_uids = set(df_new["uid"].dropna().astype(str).unique())
if SPLIT_PATH.exists():
    split_manifest = pd.read_csv(SPLIT_PATH, dtype={"uid": str})
    if not {"uid", "split"}.issubset(split_manifest.columns): raise ValueError("The manifest must have uid and split columns.")
    split_manifest = split_manifest[["uid", "split"]].copy()
    split_manifest["uid"] = split_manifest["uid"].astype(str)
    split_manifest["split"] = split_manifest["split"].astype(str).str.lower().replace({"val": "validation"})
    if split_manifest["uid"].duplicated().any(): raise ValueError("The manifest contains duplicate study IDs.")
    invalid = set(split_manifest["split"]) - set(split_names)
    if invalid: raise ValueError(f"Unexpected split labels: {sorted(invalid)}")
    extra_uids = sorted(current_uids - set(split_manifest["uid"]))
    if extra_uids:
        counts = split_manifest["split"].value_counts().reindex(split_names, fill_value=0).to_dict()
        targets = {name: split_ratios[name] * (len(split_manifest) + len(extra_uids)) for name in split_names}
        new_rows = []
        for uid in extra_uids:
            chosen = max(split_names, key=lambda name: (targets[name] - counts[name], -split_names.index(name)))
            new_rows.append({"uid": uid, "split": chosen}); counts[chosen] += 1
        split_manifest = pd.concat([split_manifest, pd.DataFrame(new_rows)], ignore_index=True)
        split_manifest.to_csv(SPLIT_PATH, index=False)
        print(f"Added {len(extra_uids)} IDs; preserved existing assignments.")
else:
    uids = np.array(sorted(current_uids)); np.random.default_rng(42).shuffle(uids)
    train_end, val_end = int(.80 * len(uids)), int(.90 * len(uids))
    split_manifest = pd.DataFrame({"uid": uids, "split": ["train"] * train_end + ["validation"] * (val_end-train_end) + ["test"] * (len(uids)-val_end)})
    split_manifest.to_csv(SPLIT_PATH, index=False)
split_map = dict(zip(split_manifest["uid"], split_manifest["split"]))
df_new = df_new.copy(); df_new["split"] = df_new["uid"].astype(str).map(split_map)
if df_new["split"].isna().any(): raise ValueError("Some rows still have no split assignment.")
train_df = df_new[df_new["split"] == "train"].copy().reset_index(drop=True)
val_df = df_new[df_new["split"] == "validation"].copy().reset_index(drop=True)
test_df = df_new[df_new["split"] == "test"].copy().reset_index(drop=True)
assert set(train_df.uid).isdisjoint(val_df.uid)
assert set(train_df.uid).isdisjoint(test_df.uid)
assert set(val_df.uid).isdisjoint(test_df.uid)
print("Split manifest:", SPLIT_PATH)
print("Studies:", split_manifest["split"].value_counts().to_dict())
print("Rows:", len(train_df), len(val_df), len(test_df))


In [ ]:
print("\nClass Distribution in Training Set:\n", train_df['class'].value_counts())
print("\nClass Distribution in Validation Set:\n", val_df['class'].value_counts())
print("\nClass Distribution in Testing Set:\n", test_df['class'].value_counts())

In [ ]:
train_df.iloc[3]

### Save splits (so a disconnect doesn't force you to redo data prep)

In [ ]:
# Local paths are configurable from the VS Code/Jupyter environment.
from pathlib import Path
import os
DATA_DIR = Path(os.environ.get("CXR_SHARED_DIR", REPO_ROOT / "cxr_shared")).expanduser()
SAVE_DIR = Path(os.environ.get("QWEN_SAVE_DIR", REPO_ROOT / "qwen3vl_cxr")).expanduser()
DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Shared data:", DATA_DIR)
print("Qwen outputs/checkpoints:", SAVE_DIR)


In [ ]:
df_new.to_csv(DATA_DIR / "dataset.csv", index=False)
train_df.to_csv(DATA_DIR / "train.csv", index=False)
val_df.to_csv(DATA_DIR / "val.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)
print("Saved dataset and split files to", DATA_DIR)


# Qwen3-VL 2B — Zero-shot baseline and QLoRA

This section uses the public `Qwen/Qwen3-VL-2B-Instruct` model. The previous MedGemma attempt was gated and is not used here.

Run the baseline first. Confirm the preview and evaluation complete before proceeding to training. Training is separated into chunks and saves its adapter and optimizer state to Drive. The test split is for final reporting; use validation data for any model-selection decisions.


## 1. Connect a local CUDA GPU, then install model packages


In [ ]:
# Stop before installing model packages if the selected local kernel has no CUDA GPU.
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. In VS Code, select a Python environment with "
        "CUDA-enabled PyTorch and an NVIDIA driver, then restart the kernel."
    )
print("Local GPU ready:", torch.cuda.get_device_name(0))
%pip -q install "transformers>=4.57.0" accelerate peft rouge-score nltk "bitsandbytes>=0.46.1" kagglehub opencv-python missingno seaborn


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Switch from the local Python kernel to a Colab hosted GPU runtime.")
print("GPU:", torch.cuda.get_device_name(0))


## Model access

Qwen3-VL is used here instead of gated MedGemma, so this notebook does not require a Hugging Face access request or token. The public base model is downloaded into the local Hugging Face cache.


## 2. Setup / reload state — run this after EVERY reconnect

In [ ]:
from pathlib import Path
import os
DATA_DIR = Path(os.environ.get("CXR_SHARED_DIR", REPO_ROOT / "cxr_shared")).expanduser()
SAVE_DIR = Path(os.environ.get("QWEN_SAVE_DIR", REPO_ROOT / "qwen3vl_cxr")).expanduser()
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen3-VL-2B-Instruct")
PROMPT_TEXT = (
    "Describe only visible chest X-ray findings. Write a concise draft with "
    "Findings and Impression sections. Do not invent findings; avoid describing "
    "anatomy that cannot be assessed confidently from this image."
)
print("Data:", DATA_DIR)
print("Qwen results:", SAVE_DIR)


### Restore the latest checkpoint ZIP when Drive is behind

The supplied archive records step 920. Run this only if the printed Drive step is lower. When prompted, upload `qwen3vl_cxr-20261028T195256Z-1-001.zip`. It restores only the adapter and optimizer/training state; old evaluation CSVs are intentionally left untouched.

In [ ]:
# Optional local checkpoint restore. Set CXR_CHECKPOINT_ZIP only when local files are behind.
import json, os, zipfile
from pathlib import Path, PurePosixPath
state_path = SAVE_DIR / "qwen_train_state.json"
current_state = json.loads(state_path.read_text()) if state_path.exists() else {"step": 0}
current_step = int(current_state.get("step", 0))
checkpoint_zip = os.environ.get("CXR_CHECKPOINT_ZIP")
print("Current local checkpoint step:", current_step)
if checkpoint_zip:
    zip_path = Path(checkpoint_zip).expanduser().resolve()
    if not zip_path.is_file(): raise FileNotFoundError(f"Checkpoint ZIP not found: {zip_path}")
    with zipfile.ZipFile(zip_path) as archive:
        state_member = "qwen3vl_cxr/qwen_train_state.json"
        optimizer_member = "qwen3vl_cxr/qwen_optimizer_state.pt"
        adapter_prefix = "qwen3vl_cxr/qwen_lora_ckpt/"
        required = {state_member, optimizer_member, adapter_prefix + "adapter_config.json", adapter_prefix + "adapter_model.safetensors"}
        missing = required - set(archive.namelist())
        if missing: raise FileNotFoundError(f"Checkpoint ZIP is missing: {sorted(missing)}")
        incoming_state = json.loads(archive.read(state_member))
        incoming_step = int(incoming_state.get("step", 0))
        if incoming_step <= current_step:
            print(f"Keeping local step {current_step}; archive step {incoming_step} is not newer.")
        else:
            root = SAVE_DIR.resolve()
            for info in archive.infolist():
                if info.is_dir() or info.filename not in required: continue
                parts = PurePosixPath(info.filename).parts[1:]
                dest = (SAVE_DIR.joinpath(*parts)).resolve()
                if not dest.is_relative_to(root): raise ValueError(f"Unsafe archive path: {info.filename}")
                dest.parent.mkdir(parents=True, exist_ok=True); dest.write_bytes(archive.read(info))
            print(f"Restored local QLoRA checkpoint step {incoming_step} from {zip_path}")
else:
    print("No external checkpoint ZIP selected; continuing with local checkpoint files.")


In [ ]:
import pandas as pd
from pathlib import Path

if "train_df" not in globals():
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    val_df = pd.read_csv(DATA_DIR / "val.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

if "image_lookup" not in globals():
    import kagglehub
    ds = Path(kagglehub.dataset_download("raddar/chest-xrays-indiana-university"))
    image_lookup = {
        p.name: p for p in ds.rglob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    }

print(len(train_df), len(val_df), len(test_df), "| images:", len(image_lookup))


## 3. Helpers (generate + evaluate, evaluation resumes from its CSV)

In [ ]:
import json
import re
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

MAX_NEW = 160

def norm_text(text):
    text = re.sub(r"[^a-z0-9\s]", " ", str(text).lower())
    return re.sub(r"\s+", " ", text).strip()

def load_img(fname):
    return Image.open(image_lookup[str(fname)]).convert("RGB")

def generate_report(model, processor, image, max_new_tokens=MAX_NEW):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT_TEXT},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    )
    inputs.pop("token_type_ids", None)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(COMPUTE_DTYPE)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(output[0, input_len:], skip_special_tokens=True).strip()


In [ ]:
def evaluate_model(model, processor, df_eval, label):
    """Resume predictions for exactly the supplied study IDs; never mix old rows."""
    path = SAVE_DIR / f"eval_{label}.csv"
    expected = set(df_eval["uid"].astype(str))
    prior = pd.read_csv(path).to_dict("records") if path.exists() else []
    rows = [r for r in prior if str(r.get("uid", "")) in expected]
    done = {str(r["uid"]) for r in rows}
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    smooth = SmoothingFunction().method1
    model.eval()
    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc=label):
        uid = str(row["uid"])
        if uid in done: continue
        prediction = generate_report(model, processor, load_img(row["filename"]))
        reference = str(row["processed_report"])
        pred_norm, ref_norm = norm_text(prediction), norm_text(reference)
        rows.append({"uid": uid, "filename": row["filename"], "prediction": prediction,
                     "reference": reference,
                     "rougeL": scorer.score(ref_norm, pred_norm)["rougeL"].fmeasure,
                     "bleu": sentence_bleu([ref_norm.split()], pred_norm.split(), smoothing_function=smooth)})
        done.add(uid)
        pd.DataFrame(rows).to_csv(path, index=False)
    result = pd.DataFrame(rows)
    if set(result["uid"].astype(str)) != expected:
        raise RuntimeError(f"Evaluation {label} is incomplete or contains unexpected study IDs.")
    metrics = {"rougeL": float(result["rougeL"].mean()), "bleu": float(result["bleu"].mean()),
               "n": len(result), "evaluation_label": label}
    (SAVE_DIR / f"metrics_{label}.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(f"{label}: ROUGE-L={metrics['rougeL']:.3f} BLEU={metrics['bleu']:.3f} n={metrics['n']}")
    return metrics


In [ ]:
# Development-only baseline: fixed sample from validation, not test.
NUM_EVAL_SAMPLES = 24
frontal_val = val_df[val_df["projection"].astype(str).str.lower().isin(["frontal", "ap", "pa"])].drop_duplicates("uid")
eval_df = frontal_val.sample(n=min(NUM_EVAL_SAMPLES, len(frontal_val)), random_state=42).reset_index(drop=True)
if eval_df.empty: raise ValueError("No frontal studies found in the validation split.")
print("Validation frontal studies:", len(eval_df))


## 4. Qwen zero-shot baseline

The baseline runs before training on a fixed subset of the held-out test studies.


In [ ]:
print("Qwen validation baseline already done:", (SAVE_DIR / "metrics_qwen_zero_shot_validation.json").exists())


In [ ]:
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before loading Qwen.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    attn_implementation="eager",
)
model.eval()
print("Qwen3-VL loaded on", torch.cuda.get_device_name(0))


In [ ]:
import gc
import torch
from transformers import AutoProcessor

# Lower the image resolution and generated report length for the T4.
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 128 * 28 * 28
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    size={"shortest_edge": MIN_PIXELS, "longest_edge": MAX_PIXELS},
)

gc.collect()
torch.cuda.empty_cache()

row = eval_df.iloc[0]
generated = generate_report(
    model,
    processor,
    load_img(row["filename"]),
    max_new_tokens=64,
)
print("Generated:\n", generated)
print("\nGround truth:\n", row["processed_report"])

In [ ]:
baseline_metrics = evaluate_model(model, processor, eval_df, "qwen_zero_shot_validation")


In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print("Released baseline model memory.")


In [ ]:
import json
import pandas as pd
metrics_path = SAVE_DIR / "metrics_qwen_zero_shot_validation.json"
predictions_path = SAVE_DIR / "eval_qwen_zero_shot_validation.csv"
print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
predictions = pd.read_csv(predictions_path)
display(predictions[["uid", "prediction", "reference", "rougeL", "bleu"]].head(5))


## 5. Load 4-bit model + LoRA (auto-resumes from the last checkpoint)

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig

if "processor" not in globals():
    processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    attn_implementation="eager",
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
print("Qwen 4-bit model loaded for LoRA.")


In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

CKPT = SAVE_DIR / "qwen_lora_ckpt"
if (CKPT / "adapter_config.json").exists():
    model = PeftModel.from_pretrained(model, str(CKPT), is_trainable=True)
    print("Resumed Qwen LoRA from", CKPT)
else:
    cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, cfg)
    print("Created Qwen LoRA adapter")
model.print_trainable_parameters()

def lora_b_norm(m):
    return sum(p.float().norm().item() for n, p in m.named_parameters() if "lora_B" in n)
print("LoRA B norm:", lora_b_norm(model))


## 6. Build Qwen training inputs and run a sanity check


In [ ]:
def build_batch(row):
    image = load_img(row["filename"])
    prompt = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT_TEXT},
    ]}]
    prompt_inputs = processor.apply_chat_template(
        prompt, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    )
    prompt_inputs.pop("token_type_ids", None)
    prompt_len = prompt_inputs["input_ids"].shape[-1]
    full_msgs = prompt + [{
        "role": "assistant",
        "content": [{"type": "text", "text": str(row["processed_report"])}],
    }]
    full = processor.apply_chat_template(
        full_msgs, add_generation_prompt=False, tokenize=True,
        return_dict=True, return_tensors="pt",
    )
    full.pop("token_type_ids", None)
    # Keep every tensor field, including Qwen's image_grid_thw.
    batch = {k: v.to(model.device) for k, v in full.items() if torch.is_tensor(v)}
    if "pixel_values" in batch:
        batch["pixel_values"] = batch["pixel_values"].to(COMPUTE_DTYPE)
    labels = batch["input_ids"].clone()
    labels[:, :prompt_len] = -100
    return batch, labels

ft_train_df = (
    train_df[train_df["projection"].astype(str).str.lower().eq("frontal")]
    .dropna(subset=["processed_report"])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
if ft_train_df.empty:
    raise ValueError("No frontal training rows with usable reports.")
print("Qwen training studies:", ft_train_df["uid"].nunique())


In [ ]:
batch, labels = build_batch(ft_train_df.iloc[0])
print("Input keys:", list(batch.keys()))
print("Sequence length:", batch["input_ids"].shape[-1])
print("Supervised tokens:", (labels != -100).sum().item())
assert "pixel_values" in batch
assert (labels != -100).sum() > 5
model.eval()
with torch.no_grad():
    print("Sanity-check loss:", model(**batch, labels=labels).loss.item())
del batch, labels
torch.cuda.empty_cache()


## 7. Continue QLoRA in short, resumable chunks

Run the training-loop cell first. One click now performs one optimizer update (4 micro-batches) and saves the adapter, optimizer, and step. Only run validation after a successful save. If the kernel restarts, rerun setup/reload and resume from local checkpoint files; set `CXR_CHECKPOINT_ZIP` only to restore a newer archive. Smaller image size lowers peak activation memory; a runtime/quota failure still cannot be fixed by notebook code alone.

In [ ]:
import bitsandbytes as bnb

# No TOTAL_STEPS cap: each run adds another checkpointed chunk.
GRAD_ACCUM, LR = 4, 1e-4
CHUNK, SAVE_EVERY = 4, 4  # one optimizer update per run; save after each update
assert CHUNK % GRAD_ACCUM == 0
assert SAVE_EVERY % GRAD_ACCUM == 0
state_path = SAVE_DIR / "qwen_train_state.json"
optimizer_path = SAVE_DIR / "qwen_optimizer_state.pt"
state = json.load(open(state_path)) if state_path.exists() else {"step": 0, "losses": []}
state.setdefault("losses", [])
optimizer = bnb.optim.AdamW8bit(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)
if optimizer_path.exists():
    optimizer.load_state_dict(torch.load(optimizer_path, map_location="cpu"))
print(
    f"Resuming at micro-step {state['step']} "
    f"(~{state['step'] // GRAD_ACCUM} optimizer updates)"
)


In [ ]:
import gc

def save_progress():
    model.save_pretrained(str(CKPT))
    torch.save(optimizer.state_dict(), optimizer_path)
    tmp_path = SAVE_DIR / "qwen_train_state.tmp"
    with open(tmp_path, "w") as f:
        json.dump(state, f)
    os.replace(tmp_path, state_path)

model.train()
model.config.use_cache = False
start = int(state["step"])
assert start % GRAD_ACCUM == 0, "Saved step is not aligned with gradient accumulation."
end = start + CHUNK
optimizer.zero_grad(set_to_none=True)
pbar = tqdm(range(start, end), desc=f"micro-steps {start}-{end}")

for step in pbar:
    batch, labels = build_batch(ft_train_df.iloc[step % len(ft_train_df)])
    try:
        loss = model(**batch, labels=labels).loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss at micro-step {step}")
        (loss / GRAD_ACCUM).backward()
        state["losses"].append(float(loss.detach().cpu()))
        pbar.set_postfix(loss=f"{loss.item():.3f}")
    finally:
        del batch, labels

    if (step + 1) % GRAD_ACCUM == 0:
        trainable = [p for p in model.parameters() if p.requires_grad]
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        state["step"] = step + 1
        save_progress()
        del trainable
        gc.collect()
        torch.cuda.empty_cache()

print("LoRA B norm:", lora_b_norm(model))
print("Mean loss last 20:", float(np.mean(state["losses"][-20:])))
print(f"Saved at micro-step {state['step']} (~{state['step'] // GRAD_ACCUM} optimizer updates).")

### Check the current checkpoint on validation data

This uses a fixed sample from the validation split and writes a new file for each training step, so it will not silently reuse the old test predictions. Use these scores to decide whether another training chunk is helping. Keep the test split for one final report.

In [ ]:
# Evaluate the just-saved checkpoint on the same validation studies each time.
model.eval()
model.config.use_cache = True
val_frontal_df = (
    val_df[val_df["projection"].astype(str).str.lower().eq("frontal")]
    .drop_duplicates("uid")
    .reset_index(drop=True)
)
val_eval_df = val_frontal_df.sample(
    n=min(24, len(val_frontal_df)), random_state=42
).reset_index(drop=True)
if val_eval_df.empty:
    raise ValueError("No frontal studies are available in the validation split.")
val_label = f"qwen_lora_validation_step_{state['step']}"
validation_metrics = evaluate_model(model, processor, val_eval_df, val_label)
print("Fresh validation files:", SAVE_DIR / f"eval_{val_label}.csv",
      "and", SAVE_DIR / f"metrics_{val_label}.json")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(pd.Series(state["losses"]).rolling(10).mean()); plt.xlabel("step"); plt.ylabel("loss (10-step avg)"); plt.show()

## 8. Test the fine-tuned model

In [ ]:
# Validation preview only. Do not inspect a held-out test reference during development.
model.eval(); model.config.use_cache = True
row = eval_df.iloc[0]
print("Generated validation draft:\n", generate_report(model, processor, load_img(row["filename"])))
print("\nValidation reference:\n", row["processed_report"])


In [ ]:
ft_metrics = evaluate_model(model, processor, eval_df, "qwen_lora_validation_current")


In [ ]:
base = json.load(open(SAVE_DIR / "metrics_qwen_zero_shot_validation.json"))
ft = json.load(open(SAVE_DIR / "metrics_qwen_lora_validation_current.json"))
display(pd.DataFrame([
    {"stage": "Qwen zero-shot validation", **base},
    {"stage": "Qwen QLoRA validation", **ft},
]))


## 9. Save the final adapter

In [ ]:
FINAL_DIR = SAVE_DIR / "qwen3vl-cxr-lora"
model.save_pretrained(str(FINAL_DIR))
processor.save_pretrained(str(FINAL_DIR))
print("Saved final adapter and processor to", FINAL_DIR)


## Final paired held-out evaluation
Run once after model selection is complete. This excludes the previously viewed test studies, evaluates both models on the exact same remaining cases, and saves resumable predictions. ROUGE/BLEU measure text overlap only; qualified clinical review is required before making accuracy claims.


In [ ]:
# Final paired test evaluation: exclude studies previously inspected in old 24-case runs.
import pandas as pd
previously_exposed = set()
for old_name in ["eval_qwen_zero_shot.csv", "eval_qwen_lora_finetuned.csv"]:
    old_path = SAVE_DIR / old_name
    if old_path.exists():
        old = pd.read_csv(old_path)
        if "uid" in old.columns: previously_exposed.update(old["uid"].astype(str))
final_test_df = test_df[test_df["projection"].astype(str).str.lower().isin(["frontal", "ap", "pa"])].drop_duplicates("uid").copy()
final_test_df = final_test_df[~final_test_df["uid"].astype(str).isin(previously_exposed)].reset_index(drop=True)
if final_test_df.empty: raise RuntimeError("No untouched frontal test studies remain after excluding prior exposures.")
print("Untouched paired test studies:", len(final_test_df), "| Previously exposed excluded:", len(previously_exposed))
final_lora_metrics = evaluate_model(model, processor, final_test_df, "qwen_lora_test_final")
del model
import gc
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
final_processor = AutoProcessor.from_pretrained(MODEL_ID, size={"shortest_edge": MIN_PIXELS, "longest_edge": MAX_PIXELS})
final_base = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, torch_dtype=COMPUTE_DTYPE,
    device_map="auto", attn_implementation="eager",
)
final_base.eval()
final_base_metrics = evaluate_model(final_base, final_processor, final_test_df, "qwen_zero_shot_test_final")
print("Final paired metrics (text overlap only):")
display(pd.DataFrame([
    {"stage": "Qwen zero-shot final test", **final_base_metrics},
    {"stage": "Qwen QLoRA final test", **final_lora_metrics},
]))
